# RAG 절차
- Indexing
- Retrieving

# 1. Indexing (인덱싱)

## 1.1. 데이터 로드

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./data/Samsung_Card_Manual_Korean_1.3.pdf")
pages = loader.load()  # List[Document] 형태로 반환

## 1.2. 텍스트 분리

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = splitter.split_documents(pages)

## 1.3. 임베딩
- Vector DB인 FAISS DB를 설치합니다.
- !pip install faiss-cpu --no-cache-dir

In [7]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

## 1.4. 벡터 저장
- indexing 완료

In [8]:
from langchain_community.vectorstores import FAISS

vectordb = FAISS.from_documents(docs, embeddings)

# 2. 검색과 생성 (Retrieval & Generation) - 추론 단계

## 2.1. 검색 (Retrieval)

In [ ]:
# from langchain_community.vectorstores import VectorStore

retriever = vectordb.as_retriever(search_kwargs={"k": 3})
# k는 반환할 청크 수입니다. 도메인과 청크 크기에 따라 조정합니다.

## 2.2. 프롬프트 구성 (Prompt Construction)

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

prompt_text = '''
    너는 삼성전자 메모리카드 매뉴얼에 대한 전문 어시스턴트이다.
    다음의 참고 문서를 바탕으로 질문에 정확하게 답하라.

    [참고문서]
    {context}

    [질문]
    {question}

    한글로 간결하고 정확하게 답변하라.
'''

prompt = ChatPromptTemplate.from_template(prompt_text)

# prompt = ChatPromptTemplate.from_messages(prompt_text)

## 2.3. 응답 생성 (Generation)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "이 유틸리티는 동시에 몇 개의 메모리카드나 UFD를 인식할 수 있나?"  # 예시 질의
answer = rag_chain.invoke(query)

In [18]:
answer

'이 유틸리티는 동시에 최대 8개의 메모리 카드나 UFD를 인식할 수 있습니다.'

In [21]:
answer = rag_chain.invoke("삼성 메모리카드 인증 방법은")
print(answer)

삼성 메모리카드 인증 방법은 다음과 같습니다:

1. 삼성 메모리 카드/UFD 인증 유틸리티를 사용합니다.
2. 인증이 가능한 메모리 카드는 V 표시가 있는 제품입니다.
3. 유틸리티를 실행한 후, 인증할 대상 드라이브를 선택합니다.
4. 인증 절차가 진행되며, 일반적으로 결과는 5초 이내에 표시됩니다.

이 유틸리티는 한국어, 영어, 중국어를 지원합니다.


In [22]:
answer = rag_chain.invoke("PC에 연결된 Card/UFD 제품이 없는 경우 어떻게 해야 하나")
print(answer)

PC에 연결된 Card/UFD 제품이 없는 경우, 유틸리티에서 "메모리 카드/USB 플래시 드라이브 삽입" 메시지가 표시됩니다. 이때 사용자가 카드/UFD를 삽입하면 유틸리티가 이를 자동으로 인식하고 인증을 진행합니다.


# 3. Chatbot 만들기

1. 사용자 질문 입력    
 ↓     
2. Streamlit UI (chat_input)    
 ↓     
3. RAG 체인 (retriever → prompt → llm → parser)     
 ↓     
4. 답변 출력 (chat_message)    
 ↓ 
5. 대화 히스토리 누적 (session_state)   

03_simple_rag_project/
├── app.py              ← Streamlit 메인
├── rag_chain.py        ← RAG 체인 빌더
├── data/
│   └── samsung_manual.pdf
└── .env